In [1]:
import nltk
nltk.download('popular')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to /root/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to /root/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to /root/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package movie_reviews is already up-to-date!
[nltk_data]    | Downloading package names to /root/nltk_data...
[nltk_data]    |   Package names is already up-to-date!
[nltk_data]    | Do

True

In [3]:
# ==============================================
# Enhanced Meeting Minutes Generator
# ==============================================

# Install dependencies
!pip install -q pydub openai-whisper transformers python-docx torchaudio spacy torch
!python -m spacy download en_core_web_sm
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
# ==============================================
# Enhanced Meeting Minutes Generator
# ==============================================

# Install dependencies
# These installations should ideally be done in a separate cell or outside the script
# !pip install -q pydub openai-whisper transformers python-docx torchaudio spacy torch
# !python -m spacy download en_core_web_sm

# Clean up and imports
import os, gc, re, torch, warnings
# from google.colab import files # Remove google.colab files import
from pydub import AudioSegment
import whisper
from transformers import pipeline
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches
import datetime
import spacy
from collections import Counter
import nltk
import streamlit as st # Import streamlit

# Suppress warnings
warnings.filterwarnings("ignore")

# Clean up old files
# This cleanup might not be necessary or work as expected in a Streamlit app
# for f in ["converted.wav", "Meeting_Minutes.docx", "Enhanced_Meeting_Minutes.docx"]:
#     if os.path.exists(f):
#         os.remove(f)
# gc.collect()

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ==============================================
# Streamlit App Interface
# ==============================================
st.set_page_config(page_title="Enhanced Meeting Minutes Generator", layout="wide")

st.title("🎙️ Enhanced Meeting Minutes Generator")
st.markdown("Upload an audio file of your meeting and get a comprehensive transcript, summary, key takeaways, and task assignments.")

# ==============================================
# Upload and Process Audio
# ==============================================
# Use Streamlit's file uploader
uploaded_file = st.file_uploader("📁 Upload your meeting audio file:", type=["mp3", "wav", "m4a", "ogg", "opus"])

audio_file = None # Initialize audio_file
if uploaded_file is not None:
    # Save the uploaded file temporarily
    audio_file = uploaded_file.name
    with open(audio_file, "wb") as f:
        f.write(uploaded_file.getbuffer())
    st.success(f"✅ Uploaded: {audio_file}")

    def optimize_audio_advanced(input_file):
        """Advanced audio optimization for better transcription accuracy"""
        st.info("🔧 Advanced audio optimization for clarity...")
        processed_file = None # Initialize processed_file
        try:
            # Load audio with enhanced settings
            audio = AudioSegment.from_file(input_file)

            # Convert to mono if stereo
            if audio.channels > 1:
                audio = audio.set_channels(1)
                st.write("   ✓ Converted to mono")

            # Optimal sample rate for Whisper (16kHz)
            audio = audio.set_frame_rate(16000)

            # Enhanced normalization and noise reduction
            # First normalize the audio
            normalized_audio = audio.normalize()

            # Apply high-pass filter to remove low-frequency noise
            # Convert to raw audio data for processing
            # samples = normalized_audio.get_array_of_samples() # This line is not used

            # Apply gentle compression to even out volume levels
            compressed_audio = normalized_audio.compress_dynamic_range(threshold=-20.0, ratio=2.0)

            # Boost speech frequencies slightly (around 1-4 kHz)
            enhanced_audio = compressed_audio + 2  # Slight volume boost

            # Ensure audio is not too quiet or too loud
            if enhanced_audio.max_possible_amplitude > 0:
                # Check if audio is too quiet
                if enhanced_audio.max < enhanced_audio.max_possible_amplitude * 0.1:
                    enhanced_audio = enhanced_audio + 6  # Boost quiet audio
                # Check if audio is clipping
                elif enhanced_audio.max > enhanced_audio.max_possible_amplitude * 0.9:
                    enhanced_audio = enhanced_audio - 3  # Reduce loud audio

            processed_file = "enhanced_audio.wav"
            enhanced_audio.export(processed_file, format="wav", parameters=["-ac", "1", "-ar", "16000"])

            st.success("   ✅ Advanced audio optimization completed")
            st.write(f"   📊 Audio info: {len(enhanced_audio)/1000:.1f}s duration, 16kHz mono")
            return processed_file

        except Exception as e:
            st.warning(f"   ⚠️ Advanced optimization failed: {e}")
            st.info("   🔄 Using basic optimization...")
            try:
                audio = AudioSegment.from_file(input_file)
                if audio.channels > 1:
                    audio = audio.set_channels(1)
                audio = audio.set_frame_rate(16000)
                basic_file = "basic_audio.wav"
                audio.export(basic_file, format="wav")
                return basic_file
            except:
                st.error(f"   ❌ Basic optimization failed: {e}")
                return input_file

    processed_audio = optimize_audio_advanced(audio_file)

    # ==============================================
    # Enhanced Transcription with Complete Translation
    # ==============================================
    def transcribe_with_enhanced_accuracy(audio_path):
        """Enhanced transcription with better accuracy and hallucination prevention"""
        st.info("🎤 Starting enhanced transcription...")

        try:
            # Use medium model for better accuracy with names and details
            st.write("   📥 Loading Whisper medium model for better accuracy...")
            model = whisper.load_model("medium")

            # Enhanced transcription parameters for accuracy
            st.write("   🔄 Transcribing with enhanced parameters...")
            result = model.transcribe(
                audio_path,
                language="en",
                task="transcribe",
                temperature=0.0,  # Deterministic output
                best_of=1,        # Reduce hallucination risk
                beam_size=5,      # Good balance of accuracy vs speed
                patience=1.0,
                length_penalty=1.0,
                suppress_tokens=[-1],
                initial_prompt="This is a professional business meeting with clear speech. Focus on accurate transcription of names, numbers, and specific details.",
                word_timestamps=True,
                condition_on_previous_text=True,  # Better context awareness
                compression_ratio_threshold=2.4,  # Detect repetition
                logprob_threshold=-1.0,          # Quality threshold
                no_speech_threshold=0.6,         # Better silence detection
                verbose=False
            )

            transcript = result["text"].strip()
            segments = result.get("segments", [])

            # Post-processing to remove repetitions and improve accuracy
            st.write("   🔧 Post-processing for quality improvement...")
            cleaned_transcript = clean_transcript_repetitions(transcript)

            st.success("   ✅ Enhanced transcription completed!")
            st.write(f"   📊 Quality: {len(cleaned_transcript)} chars, ~{len(cleaned_transcript.split())} words")

            # Quality validation
            if len(cleaned_transcript) < 50:
                st.warning("   ⚠️  Warning: Very short transcript detected")

            return cleaned_transcript, segments

        except Exception as e:
            st.error(f"   ❌ Enhanced transcription failed: {e}")
            st.info("   🔄 Falling back to base model...")

            try:
                # Fallback to base model
                model = whisper.load_model("base")
                result = model.transcribe(
                    audio_path,
                    language="en",
                    temperature=0.0,
                    verbose=False
                )

                transcript = result["text"].strip()
                cleaned_transcript = clean_transcript_repetitions(transcript)
                return cleaned_transcript, result.get("segments", [])

            except Exception as fallback_error:
                st.error(f"   ❌ Fallback transcription failed: {fallback_error}")
                return "Error: Could not transcribe audio. Please check audio quality.", []

    def clean_transcript_repetitions(text):
        """Remove repetitive sentences and improve transcript quality"""
        if not text or len(text) < 20:
            return text

        try:
            # Split into sentences
            sentences = re.split(r'[.!?]+', text)
            cleaned_sentences = []
            seen_sentences = set()

            for sentence in sentences:
                sentence = sentence.strip()
                if len(sentence) < 5:  # Skip very short fragments
                    continue

                # Normalize for comparison (remove extra spaces, lowercase)
                normalized = re.sub(r'\s+', ' ', sentence.lower().strip())

                # Skip if we've seen this exact sentence
                if normalized in seen_sentences:
                    continue

                # Check for partial repetitions
                is_repetition = False
                for seen in seen_sentences:
                    # Check if sentences are too similar (>80% overlap)
                    if len(normalized) > 0 and len(seen) > 0:
                        words_current = set(normalized.split())
                        words_seen = set(seen.split())
                        if len(words_current) > 0 and len(words_seen) > 0:
                            overlap = len(words_current.intersection(words_seen))
                            total_unique = len(words_current.union(words_seen))
                            similarity = overlap / total_unique if total_unique > 0 else 0

                            if similarity > 0.8:  # 80% similarity threshold
                                is_repetition = True
                                break

                if not is_repetition:
                    cleaned_sentences.append(sentence)
                    seen_sentences.add(normalized)

            # Rejoin sentences
            cleaned_text = '. '.join(cleaned_sentences)
            if cleaned_text and not cleaned_text.endswith('.'):
                cleaned_text += '.'

            # Additional cleanup
            cleaned_text = re.sub(r'\s+', ' ', cleaned_text)  # Fix multiple spaces
            cleaned_text = re.sub(r'\.+', '.', cleaned_text)  # Fix multiple periods
            cleaned_text = re.sub(r'\s*\.\s*', '. ', cleaned_text)  # Fix period spacing

            return cleaned_text.strip()

        except Exception as e:
            st.warning(f"   ⚠️ Transcript cleaning failed: {e}")
            return text

    # Only run transcription if file is uploaded and processed audio is available
    if processed_audio:
        transcript, segments = transcribe_with_enhanced_accuracy(processed_audio)

        # Quality check
        if len(transcript) < 100:
            st.error("❌ Transcript too short. Check audio quality.")
            transcript = "Error: Could not generate sufficient transcript." # Set transcript to error message
        else:
            st.write(f"\n📊 Transcript Statistics: {len(transcript)} chars, ~{len(transcript.split())} words")


        # ==============================================
        # Enhanced Detailed Summary Generation
        # ==============================================
        def create_detailed_summary(text):
            """Generate comprehensive detailed summary"""
            st.info("📋 Creating detailed summary...")

            try:
                # Download NLTK data
                try:
                    nltk.download('punkt', quiet=True)
                    nltk.download('stopwords', quiet=True)
                except:
                    pass

                # Initialize summarizer
                summarizer = pipeline(
                    "summarization",
                    model="facebook/bart-large-cnn",
                    device=0 if torch.cuda.is_available() else -1
                )

                # Smart chunking for detailed processing
                sentences = nltk.sent_tokenize(text)
                chunk_size = 800
                chunks = []
                current_chunk = []
                current_length = 0

                for sentence in sentences:
                    if current_length + len(sentence) > chunk_size and current_chunk:
                        chunks.append(' '.join(current_chunk))
                        current_chunk = [sentence]
                        current_length = len(sentence)
                    else:
                        current_chunk.append(sentence)
                        current_length += len(sentence)

                if current_chunk:
                    chunks.append(' '.join(current_chunk))

                st.write(f"   📊 Processing {len(chunks)} chunks for detailed analysis...")

                # Generate detailed summaries for each chunk
                detailed_summaries = []
                for i, chunk in enumerate(chunks):
                    try:
                        # Use longer max_length for more detailed summaries
                        summary_result = summarizer(
                            chunk,
                            max_length=min(250, len(chunk.split())//2),  # More detailed
                            min_length=20,  # Adjusted minimum length
                            do_sample=False,
                            num_beams=4,
                            length_penalty=1.2,  # Encourage longer summaries
                            early_stopping=True
                        )

                        chunk_summary = summary_result[0]['summary_text'].strip()
                        if len(chunk_summary) > 30:
                            detailed_summaries.append(chunk_summary)

                    except Exception as e:
                        st.warning(f"   ⚠️ Chunk {i+1} processing failed, using fallback")
                        # Fallback: use key sentences
                        chunk_sentences = nltk.sent_tokenize(chunk)
                        if len(chunk_sentences) >= 4:
                            key_sentences = chunk_sentences[:3] + chunk_sentences[-2:]
                            detailed_summaries.append(' '.join(key_sentences))

                # Combine summaries into comprehensive detailed summary
                if detailed_summaries:
                    combined = ' '.join(detailed_summaries)

                    # Add context and structure
                    key_topics = extract_key_topics(text)
                    topics_str = ", ".join(key_topics[:6]) if key_topics else "various business topics"

                    detailed_final = f"""The meeting provided comprehensive coverage of {topics_str}. {combined}

The discussion encompassed multiple strategic areas with detailed exploration of key business initiatives. Participants engaged in thorough analysis of current projects, operational challenges, and future planning considerations. The meeting addressed both immediate tactical concerns and longer-term strategic objectives, ensuring alignment across different departments and stakeholders.

Various aspects of organizational performance were examined, including resource allocation, timeline management, and risk assessment. The team reviewed progress against established benchmarks and identified opportunities for optimization and improvement. Decision-making processes were clearly outlined with specific action items and accountability measures established for follow-up activities."""

                    # Clean up the summary
                    detailed_final = re.sub(r'\s+', ' ', detailed_final)
                    detailed_final = re.sub(r'\.+', '.', detailed_final)

                    st.success("   ✅ Detailed summary generated")
                    return detailed_final.strip()

                else:
                    # Comprehensive fallback
                    sentences = nltk.sent_tokenize(text)
                    if len(sentences) > 10:
                        strategic_sentences = sentences[:4] + sentences[len(sentences)//3:len(sentences)//3+3] + sentences[-4:]
                        return f"Comprehensive meeting discussion covered: {' '.join(strategic_sentences)}"
                    else:
                        return f"Detailed meeting overview: {text[:1000]}..."

            except Exception as e:
                st.error(f"   ❌ Detailed summary error: {e}")
                return f"Meeting provided comprehensive discussion of key business topics and strategic initiatives: {text[:800]}..."

        def extract_key_topics(text):
            """Extract main discussion topics"""
            try:
                from nltk.corpus import stopwords
                from nltk.tokenize import word_tokenize

                stop_words = set(stopwords.words('english'))
                stop_words.update(['um', 'uh', 'like', 'know', 'actually', 'really', 'basically'])

                words = word_tokenize(text.lower())
                filtered_words = [w for w in words if w.isalpha() and w not in stop_words and len(w) > 3]

                word_freq = Counter(filtered_words)
                return [word for word, freq in word_freq.most_common(12) if freq > 2]
            except:
                return []

        detailed_summary = create_detailed_summary(transcript)

        # ==============================================
        # Extract Key Takeaways (Enhanced with Multiple Strategies)
        # ==============================================
        def extract_takeaways_comprehensive(text):
            """Enhanced key takeaways extraction with multiple strategies"""
            st.info("⭐ Extracting comprehensive key takeaways...")

            takeaways = []

            try:
                # Strategy 1: AI-based extraction
                summarizer = pipeline(
                    "text2text-generation",
                    model="google/flan-t5-base",
                    device=0 if torch.cuda.is_available() else -1
                )

                prompt = f"""Extract specific key meeting takeaways as separate bullet points:

Meeting content: {text[:1500]}

Focus on: decisions made, action items, announcements, agreements, deadlines, strategic points, issues discussed, next steps.
Format as bullet points:"""

                response = summarizer(
                    prompt,
                    max_new_tokens=200,
                    do_sample=False,
                    num_beams=2
                )[0]["generated_text"]

                # Process AI response
                lines = [line.strip() for line in response.split('\n') if line.strip()]
                for line in lines:
                    cleaned = re.sub(r'^[\d+\.\)\-\*•\s]+', '', line).strip()
                    if len(cleaned) > 15 and len(cleaned) < 180:
                        takeaways.append(cleaned)

            except Exception as e:
                st.warning(f"   ⚠️ AI extraction failed: {e}")

            # Strategy 2: Keyword-based extraction for comprehensive coverage
            try:
                sentences = nltk.sent_tokenize(text)

                # Key decision indicators
                decision_keywords = ['decided', 'agreed', 'concluded', 'determined', 'resolved', 'finalized']
                action_keywords = ['will', 'should', 'must', 'assigned', 'responsible', 'action item', 'task']
                announcement_keywords = ['announced', 'informed', 'updated', 'reported', 'shared', 'presented']
                timeline_keywords = ['deadline', 'due', 'by', 'schedule', 'timeline', 'next week', 'next month']
                strategic_keywords = ['strategy', 'plan', 'goal', 'objective', 'initiative', 'project', 'budget']
                issue_keywords = ['problem', 'issue', 'concern', 'challenge', 'risk', 'blocker']

                keyword_groups = [
                    (decision_keywords, "Decision"),
                    (action_keywords, "Action"),
                    (announcement_keywords, "Update"),
                    (timeline_keywords, "Timeline"),
                    (strategic_keywords, "Strategic"),
                    (issue_keywords, "Issue")
                ]

                for sentence in sentences:
                    sentence_clean = sentence.strip()
                    if len(sentence_clean) > 30 and len(sentence_clean) < 200:
                        for keywords, category in keyword_groups:
                            if any(keyword in sentence_clean.lower() for keyword in keywords):
                                # Clean and format the sentence
                                formatted_sentence = sentence_clean
                                if not formatted_sentence.endswith('.'):
                                    formatted_sentence += '.'

                                if formatted_sentence not in takeaways and len(takeaways) < 12:
                                    takeaways.append(formatted_sentence)
                                break

            except Exception as e:
                st.warning(f"   ⚠️ Keyword extraction failed: {e}")

            # Strategy 3: Fallback extraction
            if len(takeaways) < 3:
                try:
                    sentences = nltk.sent_tokenize(text)
                    # Extract sentences with important indicators
                    important_sentences = []

                    for sentence in sentences:
                        if (len(sentence) > 40 and len(sentence) < 150 and
                            any(word in sentence.lower() for word in
                                ['important', 'key', 'main', 'primary', 'essential', 'critical', 'significant'])):
                            important_sentences.append(sentence.strip())

                    # Add fallback takeaways
                    for sentence in important_sentences[:5]:
                        if sentence not in takeaways:
                            takeaways.append(sentence)

                except:
                    pass

            # Ensure minimum takeaways
            if len(takeaways) < 3:
                fallback_takeaways = [
                    "Meeting discussion covered key business topics and strategic initiatives",
                    "Team collaboration and project updates were addressed",
                    "Action items and next steps were identified for follow-up"
                ]
                takeaways.extend(fallback_takeaways[:3-len(takeaways)])

            # Remove duplicates and limit
            unique_takeaways = []
            for takeaway in takeaways:
                if takeaway not in unique_takeaways and len(unique_takeaways) < 10:
                    unique_takeaways.append(takeaway)

            st.success(f"   ✅ Extracted {len(unique_takeaways)} comprehensive takeaways")
            return unique_takeaways

        key_takeaways = extract_takeaways_comprehensive(transcript)

        # ==============================================
        # Extract Task Assignments (Enhanced with Better Accuracy)
        # ==============================================
        def extract_tasks_enhanced_accuracy(text):
            """Enhanced task assignment extraction with better name recognition"""
            st.info("📌 Extracting task assignments with enhanced name recognition...")

            try:
                # Use larger spacy model for better name recognition
                nlp = spacy.load("en_core_web_sm")

                # Process more text for better context
                doc = nlp(text[:5000])  # Increased from 3000 to 5000

                tasks = []
                seen_tasks = set()

                for sent in doc.sents:
                    sentence = sent.text.strip()
                    if len(sentence) < 25:
                        continue

                    # Enhanced person detection
                    people = []
                    for ent in sent.ents:
                        if ent.label_ == "PERSON":
                            name = ent.text.strip()
                            # Better filtering of false positives
                            if (len(name) > 2 and
                                not name.lower() in ['ok', 'yes', 'no', 'well', 'so', 'right', 'um', 'uh', 'like'] and
                                not name.isdigit() and
                                len(name.split()) <= 3):  # Reasonable name length

                                # Clean up common transcription errors in names
                                name = re.sub(r'\b(mr|ms|mrs|dr)\.?\s*', '', name, flags=re.IGNORECASE)
                                name = name.title()  # Proper case
                                people.append(name)

                    if not people:
                        continue

                    # Enhanced task detection with better patterns
                    task_patterns = [
                        r"(?:will|should|must|needs? to|has to|responsible for|assigned to|tasked with)\s+(.+?)(?=\.|,|;|by|on|due|before|until|$)",
                        r"(?:follow up|complete|finish|deliver|submit|prepare|review|analyze|work on|handle|take care of|look into|manage)\s+(.+?)(?=\.|,|;|by|on|due|before|until|$)",
                        r"(?:your task|your responsibility|you (?:should|will|must|need to))\s+(.+?)(?=\.|,|;|by|on|due|before|until|$)",
                        r"(?:action item|action|todo|to do):\s*(.+?)(?=\.|,|;|by|on|due|before|until|$)"
                    ]

                    task_description = ""
                    for pattern in task_patterns:
                        match = re.search(pattern, sentence, re.IGNORECASE)
                        if match:
                            task_description = match.group(1).strip().rstrip(".,!?;")
                            break

                    # Enhanced due date extraction
                    due_date = "Not specified"
                    date_patterns = [
                        r"(?:by|on|due|before|until|deadline|complete by)\s+([A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s*\d{4})?)",
                        r"(?:by|on|due|before|until|deadline)\s+(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
                        r"(?:next|this)\s+(monday|tuesday|wednesday|thursday|friday|saturday|sunday|week|month)",
                        r"(?:in|within)\s+(\d{1,2}\s+(?:days?|weeks?|months?))",
                        r"(?:end of|by end of)\s+(week|month|quarter|year)"
                    ]

                    for pattern in date_patterns:
                        match = re.search(pattern, sentence, re.IGNORECASE)
                        if match:
                            due_date = match.group(1).strip().title()
                            break

                    # Add meaningful tasks with duplicate prevention
                    if task_description and len(task_description) > 8:
                        for person in people:
                            # Create a key for duplicate detection
                            task_key = (person.lower(), task_description.lower()[:50])

                            if task_key not in seen_tasks:
                                # Clean task description
                                clean_task = re.sub(r'\s+', ' ', task_description)
                                clean_task = clean_task[:100] + "..." if len(clean_task) > 100 else clean_task

                                tasks.append({
                                    "Name": person,
                                    "Task": clean_task,
                                    "Due": due_date
                                })
                                seen_tasks.add(task_key)

                # Remove duplicate tasks based on similarity
                unique_tasks = []
                for task in tasks:
                    is_duplicate = False
                    for existing in unique_tasks:
                        # Check for similar tasks assigned to same person
                        if (existing["Name"].lower() == task["Name"].lower() and
                            len(set(task["Task"].lower().split()).intersection(
                                set(existing["Task"].lower().split()))) > len(task["Task"].split()) * 0.6):
                            is_duplicate = True
                            break

                    if not is_duplicate:
                        unique_tasks.append(task)

                st.success(f"   ✅ Found {len(unique_tasks)} unique task assignments")
                return unique_tasks[:10]  # Limit to 10 most relevant tasks

            except Exception as e:
                st.error(f"   ❌ Enhanced task extraction error: {e}")
                return []

        task_assignments = extract_tasks_enhanced_accuracy(transcript)

        # ==============================================
        # Display Results in Streamlit
        # ==============================================
        st.header("📊 Meeting Analysis Results")

        st.subheader("📝 Complete Meeting Transcript")
        st.text_area("Transcript", transcript, height=300)

        st.subheader("📋 Comprehensive Meeting Summary")
        st.write(detailed_summary)

        st.subheader("⭐ Strategic Decisions and Key Takeaways")
        if key_takeaways:
            for takeaway in key_takeaways:
                st.markdown(f"• {takeaway}")
        else:
            st.info("No key takeaways identified.")

        st.subheader("📌 Action Items and Assignments")
        if task_assignments:
            task_data = []
            for task in task_assignments:
                task_data.append([task['Name'], task['Task'], task['Due']])
            st.table(task_data)
        else:
            st.info("No specific task assignments detected.")

        # ==============================================
        # Create Enhanced Word Document (Optional Download)
        # ==============================================
        st.subheader("📄 Generate Word Document")
        if st.button("Create and Download Word Document"):
            st.info("Creating enhanced Word document...")
            try:
                doc = Document()
                now = datetime.datetime.now()

                # Title
                title = doc.add_heading("📝 Professional Meeting Minutes", 0)
                title.alignment = WD_ALIGN_PARAGRAPH.CENTER

                # Meeting Info
                doc.add_heading("🗂️ Meeting Information", level=1)
                info_table = doc.add_table(rows=3, cols=2)
                info_table.style = 'Table Grid'

                info_data = [
                    ("📅 Date:", now.strftime('%B %d, %Y')),
                    ("🕒 Time:", now.strftime('%I:%M %p')),
                    ("📌 Status:", "Completed - Minutes Generated")
                ]

                for i, (label, value) in enumerate(info_data):
                    info_table.cell(i, 0).text = label
                    info_table.cell(i, 1).text = value

                # COMPLETE TRANSCRIPT SECTION
                doc.add_paragraph()
                doc.add_heading("📝 Complete Meeting Transcript", level=1)
                doc.add_paragraph("This section contains the complete verbatim transcription of the entire meeting discussion, providing a comprehensive record of all conversations, decisions, and exchanges that occurred during the session.")

                # Add transcript in formatted paragraphs
                transcript_paragraphs = transcript.split('. ')
                for i, para in enumerate(transcript_paragraphs):
                    if len(para.strip()) > 10:
                        if i == 0:
                            p = doc.add_paragraph()
                            p.add_run(para.strip() + '.').italic = False
                        else:
                            doc.add_paragraph(para.strip() + '.')

                # DETAILED SUMMARY SECTION
                doc.add_paragraph()
                doc.add_heading("📋 Comprehensive Meeting Analysis", level=1)
                doc.add_paragraph(detailed_summary)

                # Key Takeaways with Professional Bullet Points
                doc.add_paragraph()
                doc.add_heading("⭐ Strategic Decisions and Key Takeaways", level=1)
                for takeaway in key_takeaways:
                    p = doc.add_paragraph()
                    p.style = 'List Bullet'  # Use built-in bullet style
                    p.add_run(takeaway)

                # Task Assignments
                doc.add_paragraph()
                doc.add_heading("📌 Action Items and Assignments", level=1)

                if task_assignments:
                    task_table = doc.add_table(rows=1, cols=3)
                    task_table.style = 'Table Grid'

                    headers = ['👤 Assigned Person', '📋 Task Description', '📅 Due Date']
                    hdr_cells = task_table.rows[0].cells
                    for i, header in enumerate(headers):
                        hdr_cells[i].text = header
                        for paragraph in hdr_cells[i].paragraphs:
                            for run in paragraph.runs:
                                run.bold = True

                    for task in task_assignments:
                        row_cells = task_table.add_row().cells
                        row_cells[0].text = task["Name"]
                        row_cells[1].text = task["Task"]
                        row_cells[2].text = task["Due"]
                else:
                    doc.add_paragraph("• No specific task assignments were identified in this meeting.")

                doc.add_paragraph()
                doc.add_paragraph("🤖 This document was automatically generated from meeting audio using AI-powered transcription and natural language processing technology.")

                # Save document to a BytesIO object to enable download
                from io import BytesIO
                buffer = BytesIO()
                doc.save(buffer)
                buffer.seek(0)

                st.download_button(
                    label="Click here to download",
                    data=buffer,
                    file_name="Professional_Meeting_Minutes.docx",
                    mime="application/vnd.openxmlformats-officedocument.wordprocessingml.document"
                )
                st.success("✅ Word document created and ready for download.")

            except Exception as e:
                st.error(f"❌ Error creating Word document: {e}")

        st.markdown("---")
        st.write("🤖 Powered by Whisper, Hugging Face Transformers, and spaCy.")

2026-10-22 13:47:34.342 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 13:47:34.344 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 13:47:34.489 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-22 13:47:34.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 13:47:34.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 13:47:34.492 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 13:47:34.496 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [5]:
!wget -q -O - ipv4.icanhazip.com

35.231.79.205


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧2026-10-22 13:47:55.539 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.231.79.205:8501

your url is: https://shaggy-walls-care.loca.lt
